# Invoice Intelligence on free Colab
Private notebook, free CPU runtime (no GPU, 2 vCPU). The setup cell prints the exact code hash of this copy.

**To use it on your own invoice:** run sections 1 and 2 cell by cell (use each cell's play button, not Run all, because cell 2b waits for a file), then open the review screen (cell 2a) and upload a JPG, PNG or PDF. The reader (OCR) takes roughly 5 seconds per page on this CPU. Check what is flagged, correct the fields, press Save review, export CSV. If the embedded screen does not load, use cell 2b (upload, extract, print) and cell 2c (download CSV).

Your uploads stay on this temporary Colab machine (`/content`) and are lost when the runtime recycles, so download the CSV before you leave. No public tunnel or share link is created.

**Section 3** (tests, evaluation, API check) is evidence for the project. Test data there is developer-generated synthetic invoices; the model probe also used 10 public sample invoices. Those are small dev checks, not an independent benchmark.

**Section 4** is an experimental slow model extractor. It is not connected to the review screen and it always needs review.


## 1. Set up

In [ ]:
!python --version; echo ---; nproc; echo ---; free -m; echo ---; df -h /content | tail -1; echo ---; (nvidia-smi || echo 'no GPU in this runtime'); echo ---; cat /proc/cpuinfo | grep 'model name' | head -1

In [ ]:
# Get the project code. Clones the public repo; if you have a zip of it instead, upload it and unzip to /content/invoice-intelligence.
!rm -rf /content/invoice-intelligence && git clone --depth 1 https://github.com/debasis009-deb/invoice-intelligence /content/invoice-intelligence
!ls /content/invoice-intelligence | head -20


In [ ]:
%%bash
# Colab now runs Python 3.13; pinned rapidocr-onnxruntime 1.4.4 has no 3.13 wheel, so use a Python 3.11 venv via uv.
cd /content/invoice-intelligence
(apt-get install -y -qq tesseract-ocr > /tmp/apt.log 2>&1 && tesseract --version | head -1) || echo 'tesseract install failed'
pip install -q uv 2>&1 | tail -1
uv venv --clear --python 3.11 /content/venv 2>&1 | tail -2
export PATH=/content/venv/bin:$PATH
uv pip install -q --python /content/venv/bin/python -r requirements-dev.txt 2>&1 | tail -3
python --version
PYTHONPATH=. python -c "import invoice_ai.evaluate as e; print('code hash', e.code_version())"


## 2. Use it on your own invoice

In [ ]:
# 2a. Start the app inside this Colab runtime and show the review screen below.
# serve_kernel_port_as_iframe is Colab's own signed-in proxy: only your logged-in browser can see it.
import subprocess, os, time, urllib.request
def healthy():
    try:
        return urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=2).status == 200
    except Exception:
        return False
if not healthy():
    env = dict(os.environ, PYTHONPATH="/content/invoice-intelligence", INVOICE_DB="/content/reviews.db")
    srv = subprocess.Popen(["/content/venv/bin/python", "-m", "invoice_ai"], cwd="/content/invoice-intelligence",
                           env=env, stdout=open("/content/app.log", "w"), stderr=subprocess.STDOUT)
    for _ in range(90):
        if healthy(): break
        time.sleep(1)
print("app healthy:", healthy())
if not healthy(): print(open("/content/app.log").read()[-2000:])
from google.colab import output
output.serve_kernel_port_as_iframe(8000, path="/ui/", height=950)


In [ ]:
# 2b. Fallback without the screen: upload files, see extraction and flags (needs cell 2a run first).
from google.colab import files
import requests, json
for name, data in files.upload().items():
    r = requests.post("http://127.0.0.1:8000/extract", files={"file": (name, data)})
    print("=====", name, "HTTP", r.status_code)
    if r.ok:
        out = r.json()
        print("review_id", out["review_id"], "| needs_review", out["needs_review"])
        for i in out["issues"]: print(" -", i["code"], ":", i["message"])
        print(json.dumps(out["invoice"], indent=1)[:3000])
    else:
        print(r.text[:500])


In [ ]:
# 2c. Download all saved reviews as CSV (works after saving in the screen).
import requests
from google.colab import files
r = requests.get("http://127.0.0.1:8000/reviews/export.csv"); r.raise_for_status()
open("/content/reviews.csv", "wb").write(r.content); print(len(r.content), "bytes")
files.download("/content/reviews.csv")


## 3. Project checks (tests, evaluation, API)

In [ ]:
%%bash
export PATH=/content/venv/bin:$PATH
cd /content/invoice-intelligence
PYTHONPATH=. timeout 1500 python -m pytest -q tests 2>&1 | tail -8

In [ ]:
%%bash
export PATH=/content/venv/bin:$PATH
cd /content/invoice-intelligence
cat > scripts/timed.py <<'PYEOF'
import resource, subprocess, sys, time
t = time.time(); r = subprocess.run(sys.argv[1:]);
print(f"[timed] seconds={time.time()-t:.1f} peak_rss_mb={resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss/1024:.0f} exit={r.returncode}")
PYEOF
export PYTHONPATH=.
python -m invoice_ai.generate 2>&1 | tail -2
python scripts/timed.py python -m invoice_ai.evaluate --split holdout --engines oracle rapidocr  | tail -12


In [ ]:
%%bash
export PATH=/content/venv/bin:$PATH
cd /content/invoice-intelligence
export PYTHONPATH=.
python - <<'EOF'
import glob
from fastapi.testclient import TestClient
from invoice_ai.api import app
f = sorted(glob.glob('data/holdout/*.jpg') + glob.glob('data/holdout/*.png'))[0]
c = TestClient(app)
r = c.post('/extract', files={'file': open(f,'rb')})
j = r.json(); print(f, r.status_code, list(j)[:8]); print(str(j)[:600])
print(c.get('/reviews').status_code)
EOF

## 4. Experimental slow model extractor (optional, not in the review screen)
CPU speed here: about 4 s for 8 tokens; a 10-document probe did not finish in 420 s. Results need review.

A full run of this model on the 75 held-out synthetic test documents was done outside Colab (2 CPUs, `scripts/model_eval.py`, about 10 s per document). It was worse than the geometry extractor on header fields (all five correct: 36.0% vs 85.3%), so it stays an experiment. See the README for the table and the agreement check.


In [ ]:
%%bash
export PATH=/content/venv/bin:$PATH
# Optional model extractor (API-only experiment). Downloads a 1.1 GB Apache-2.0 model file from Hugging Face.
cd /content/invoice-intelligence
uv pip install -q --python /content/venv/bin/python llama-cpp-python==0.3.36 --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cpu --only-binary=:all: 2>&1 | tail -2
python -c "import llama_cpp; print('llama_cpp', llama_cpp.__version__)"
curl -sL -o /content/qwen.gguf https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/qwen2.5-1.5b-instruct-q4_k_m.gguf
ls -la /content/qwen.gguf

In [ ]:
%%bash
export PATH=/content/venv/bin:$PATH
# One timed model call first, to see speed and CPU features before the long probe.
grep -o -m1 -wE 'avx2|avx512f|fma' /proc/cpuinfo | sort -u | tr '\n' ' '; echo
python - <<'EOF'
import time
from llama_cpp import Llama
t = time.time(); m = Llama(model_path='/content/qwen.gguf', n_ctx=2048, n_threads=2, verbose=False); print('load s', round(time.time()-t, 1), flush=True)
t = time.time(); o = m.create_chat_completion(messages=[{'role': 'user', 'content': 'Say OK.'}], max_tokens=8); print('8 tokens s', round(time.time()-t, 1), o['choices'][0]['message']['content'], flush=True)
EOF

In [ ]:
%%bash
export PATH=/content/venv/bin:$PATH
cd /content/invoice-intelligence
cat > scripts/timed.py <<'PYEOF'
import resource, subprocess, sys, time
t = time.time(); r = subprocess.run(sys.argv[1:]);
print(f"[timed] seconds={time.time()-t:.1f} peak_rss_mb={resource.getrusage(resource.RUSAGE_CHILDREN).ru_maxrss/1024:.0f} exit={r.returncode}")
PYEOF
export PYTHONPATH=. INVOICE_MODEL_PATH=/content/qwen.gguf
python scripts/timed.py python scripts/compare_probe.py 420 | tail -25
free -m | head -2